# Week 6 - Text Representation II : Distributed Representation (2/2)
**Notebook 2 : Sentence / Paragraph / Document-level Embeddings**

**Unstructured Data Analysis (2026-2)** · Professor: Misuk Kim · Teaching Assistant: Hojin Son

Notebook 1 gave every **word** a dense vector. Chapter 4 of the lecture asks the obvious next question:

> *If we can embed words, why not sentences, phrases, or documents?*

| Lecture slide | What we do in code |
|---|---|
| Document embedding (motivation) | two baselines: the Week 4 TF-IDF vector and the **average** of word vectors |
| Paragraph Vector: PV-DM | `Doc2Vec(dm=1)`, the paragraph id joins the context words to predict the next word |
| Paragraph Vector: PV-DBOW | `Doc2Vec(dm=0)`, the paragraph id alone predicts words sampled from the paragraph |
| "PV-DM alone usually works well, but the combination is recommended" | concatenate the two vectors |
| SPV results table (BOW vs PV-DM vs PV-DBOW) | compare all representations with one simple measurement |

> **How to use this notebook in Colab**: `File ▸ Save a copy in Drive`, then run the cells from top to bottom (`Shift + Enter`).

## &nbsp;0. Setup
Same corpus as Notebook 1, but this time one **document = one review**, and each review keeps its label (`pos` / `neg`).
The label is never used for training; we only use it at the end to check whether similar documents really are similar.

In [ ]:
!pip install -q gensim

import nltk
for r in ['movie_reviews', 'stopwords']:
    nltk.download(r, quiet=True)

In [ ]:
import numpy as np
import pandas as pd
from nltk.corpus import movie_reviews, stopwords
from gensim.models import Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

fileids = movie_reviews.fileids()
labels = np.array([movie_reviews.categories(f)[0] for f in fileids])
docs = [[w.lower() for w in movie_reviews.words(f) if w.isalpha()] for f in fileids]
english_stops = set(stopwords.words('english'))

print('#documents       :', len(docs))
print('#labels          :', dict(zip(*np.unique(labels, return_counts=True))))
print('avg tokens / doc :', round(np.mean([len(d) for d in docs])))
print('doc 0            :', ' '.join(docs[0][:25]), '...')

In [ ]:
def show_neighbors(sim_row, query, k=3):
    # print the k most similar documents to `query`, given one row of similarities
    order = [i for i in np.argsort(-sim_row) if i != query][:k]
    print(f'QUERY  [{labels[query]}] ', ' '.join(docs[query][:18]), '...')
    for rank, i in enumerate(order, 1):
        print(f'{rank}. {sim_row[i]:.3f} [{labels[i]}]', ' '.join(docs[i][:18]), '...')

QUERY = 1                 # a negative review of the film 'Virus'; feel free to change it

## &nbsp;1. Baseline 1 : TF-IDF (Week 4)
A document is already a vector in the Week 4 sense: one sparse TF-IDF row.

In [ ]:
tfidf = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
                        stop_words=list(english_stops), min_df=2)
X_tfidf = tfidf.fit_transform([' '.join(d) for d in docs])
print('TF-IDF matrix:', X_tfidf.shape, f' density {X_tfidf.nnz / np.prod(X_tfidf.shape):.2%}')

show_neighbors(cosine_similarity(X_tfidf[QUERY], X_tfidf)[0], QUERY)

## &nbsp;2. Baseline 2 : the average of the word vectors
The simplest document embedding: look up the vector of every word in the document and take the **mean**.
It is dense and short (100 numbers), but the word order is gone again, exactly as in a bag of words.

In [ ]:
# CBOW is enough here and trains in about 10 seconds
w2v = Word2Vec(sentences=docs, vector_size=100, window=5, min_count=5, sg=0, workers=4, epochs=5, seed=0)

def average_vector(tokens, wv):
    vecs = [wv[w] for w in tokens if w in wv.key_to_index and w not in english_stops]
    return np.mean(vecs, axis=0) if vecs else np.zeros(wv.vector_size)

X_avg = np.vstack([average_vector(d, w2v.wv) for d in docs])
print('average-vector matrix:', X_avg.shape)

show_neighbors(cosine_similarity(X_avg[QUERY:QUERY + 1], X_avg)[0], QUERY)

**🤔 Quick check** Why are the similarity scores all so high (0.9+)? Averaging hundreds of word vectors pulls every
review towards the same "average movie-review vector"; what remains is a small difference around that center.

## &nbsp;3. Paragraph Vector : PV-DM (Distributed Memory)
The paragraph gets its own vector, stored in a **paragraph matrix $D$** and looked up by a *paragraph id*.
Together with the word vectors of the context it predicts the next word:

* the paragraph vector is shared by all windows of the **same** paragraph, but not across paragraphs
* the word vectors are shared across **all** paragraphs

In gensim, each document becomes a `TaggedDocument(words, tags)`; the tag is the paragraph id.

In [ ]:
tagged = [TaggedDocument(words=d, tags=[i]) for i, d in enumerate(docs)]
print(tagged[0].tags, tagged[0].words[:10])

| parameter | meaning |
|---|---|
| `dm` | `1` = PV-DM, `0` = PV-DBOW |
| `vector_size` | dimension of the paragraph (and word) vectors |
| `window` | context window of PV-DM |
| `dm_mean` | PV-DM: `1` averages the paragraph and word vectors (default here), `0` sums them |
| `dm_concat` | PV-DM: `1` concatenates them instead (the slide's *Concatenate*; much slower, not used here) |
| `dbow_words` | PV-DBOW only: `1` also trains word vectors skip-gram style, `0` does not |
| `epochs` | Doc2Vec needs more passes than Word2Vec (10 - 20) |

In [ ]:
# about 30 seconds
pv_dm = Doc2Vec(tagged, dm=1, vector_size=100, window=5, min_count=5,
                epochs=20, workers=4, seed=0)

print('paragraph matrix D :', pv_dm.dv.vectors.shape)     # one row per document
print('word matrix W      :', pv_dm.wv.vectors.shape)     # one row per word

show_neighbors(cosine_similarity(pv_dm.dv.vectors[QUERY:QUERY + 1], pv_dm.dv.vectors)[0], QUERY)

### &nbsp;3-1. A vector for a new document
A document that was not in training has no row in $D$. `infer_vector` creates one by **gradient descent with all
other weights frozen**, so the result is slightly random. Run the cell twice and compare.

In [ ]:
new_review = ('the plot is predictable and the jokes fall flat , '
              'but the lead actress gives a surprisingly warm performance').split()

v_new = pv_dm.infer_vector(new_review, epochs=50)
for doc_id, score in pv_dm.dv.most_similar([v_new], topn=3):
    print(f'{score:.3f} [{labels[doc_id]}]', ' '.join(docs[doc_id][:20]), '...')

## &nbsp;4. Paragraph Vector : PV-DBOW (Distributed Bag of Words)
The context words are **ignored** in the input; the paragraph vector alone must predict words randomly sampled from
the paragraph. The slide adds: *"Does not need word vectors"*. Let us check that literally.

In [ ]:
pv_dbow = Doc2Vec(dm=0, dbow_words=0, vector_size=100, min_count=5,
                  epochs=20, workers=4, seed=0)
pv_dbow.build_vocab(tagged)

words_before = pv_dbow.wv.vectors.copy()          # .copy() matters: without it we keep a reference, not a snapshot
pv_dbow.train(tagged, total_examples=pv_dbow.corpus_count, epochs=pv_dbow.epochs)

print('word vectors unchanged by training:', np.allclose(words_before, pv_dbow.wv.vectors))
print("similarity('good', 'great') in PV-DBOW :", round(float(pv_dbow.wv.similarity('good', 'great')), 3))
print("similarity('good', 'great') in PV-DM   :", round(float(pv_dm.wv.similarity('good', 'great')), 3))

The word matrix of PV-DBOW is still the random initialization: training never touched it, yet the paragraph vectors are trained.
(`dbow_words=1` would train word vectors as well, at the cost of time.)

In [ ]:
show_neighbors(cosine_similarity(pv_dbow.dv.vectors[QUERY:QUERY + 1], pv_dbow.dv.vectors)[0], QUERY)

**🤔 Quick check** Compare the three neighbor lists for `QUERY = 1`. TF-IDF finds other reviews of the **same film**
(they share the rare word *virus*). The paragraph vectors usually rank first another review **by the same writer**
(*"the happy bastard's quickie review ..."*), even though it is about a different film:
predicting the words of a paragraph captures how it is written as much as what it is about.

## &nbsp;5. Which representation puts similar documents together?
The lecture's SPV table compares BOW-TF, BOW-TFIDF, PV-DM and PV-DBOW on IMDB, Yelp, Amazon and 20news.
We do a much simpler version on our 2,000 reviews, without training any classifier (that is Week 10):

> For every document, take its 10 nearest documents by cosine similarity and count how many of them carry the
> **same label**. 0.5 is what random neighbors would give.

The combination PV-DM + PV-DBOW, recommended on the slide, is the two vectors **concatenated** (each l2-normalized first).

In [ ]:
def neighbor_agreement(X, k=10):
    S = cosine_similarity(X)
    np.fill_diagonal(S, -np.inf)                      # a document is not its own neighbor
    nn = np.argsort(-S, axis=1)[:, :k]
    return (labels[nn] == labels[:, None]).mean()

def l2(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True)

reps = {
    'TF-IDF (Week 4)':     X_tfidf,
    'Average Word2Vec':    X_avg,
    'PV-DM':               pv_dm.dv.vectors,
    'PV-DBOW':             pv_dbow.dv.vectors,
    'PV-DM + PV-DBOW':     np.hstack([l2(pv_dm.dv.vectors), l2(pv_dbow.dv.vectors)]),
}

result = pd.DataFrame({'dimension': [X.shape[1] for X in reps.values()],
                       'same-label @10': [neighbor_agreement(X) for X in reps.values()]},
                      index=reps.keys())
result.round(3)

**🤔 Quick check**
* All five are above 0.5, so all of them put reviews of the same polarity somewhat together, without ever seeing a label.
* TF-IDF needs ~24,000 dimensions; the paragraph vectors do about as well with 100.
* On 2,000 documents PV-DBOW usually beats PV-DM. The lecture's table shows the same tendency (PV-DBOW > PV-DM on IMDB and Yelp),
  although it measures classification accuracy, a different metric, so only the ordering can be compared, not the numbers:
  PV-DBOW does not have to learn good word vectors first, which matters when the corpus is small.
* Your exact numbers vary from run to run (`workers=4`); the ordering is what to look at.

---
### ✅ What we practiced
* Two baselines for a document vector: TF-IDF (sparse, Week 4) and the average of word vectors (dense, order-free)
* PV-DM : paragraph id + context words → next word; paragraph matrix $D$ and word matrix $W$
* `infer_vector` : a vector for an unseen document by gradient descent with the model frozen
* PV-DBOW : paragraph id alone → sampled words; the word vectors are not even trained
* A comparison of all representations without training a classifier (labels used only to check the neighbors)

**Next week (Week 7)**: the 100-dimensional vectors of today are reduced to 2-D to be looked at (t-SNE), and the
24,000-dimensional TF-IDF matrix is compressed with LSA (SVD), after lecture 6-2 / 7-1 *Dimensionality Reduction*.